# TAKTKRONE-I Training Guide

Fine-tune TAKTKRONE-I on your own OCC data

## Preparation

Prepare your training data in JSONL format:

In [ ]:
import json
from pathlib import Path

# Example OCC dialogue
dialogue_example = {
    "dialogue_id": "dial_001",
    "incident_type": "signal_failure",
    "difficulty": "medium",
    "turns": [
        {"role": "operator", "content": "Signal failure at Track 1 Station A"},
        {"role": "dispatcher", "content": "Copy. Maintenance dispatched. Hold your train."},
    ],
    "ground_truth_actions": [
        "dispatch_maintenance",
        "notify_passengers",
        "monitor_situation"
    ]
}

# Write example to JSONL
Path("training_data.jsonl").write_text(
    json.dumps(dialogue_example) + "\n"
)
print("Training data prepared!")


## Load and Inspect Data

In [ ]:
from occlm.training.data_loader import OCCDataLoader

# Load data
loader = OCCDataLoader()
train_dataset, val_dataset, _ = loader.load_datasets(
    train_path="training_data.jsonl",
    val_path="training_data.jsonl"  # Use same for demo
)

# Get stats
stats = loader.get_statistics()
print(f"Train examples: {stats['train_count']}")
print(f"Val examples: {stats['val_count']}")
print(f"Avg dialogue length: {stats['avg_dialogue_length']:.0f} tokens")


## Train SFT Model

Full parameter fine-tuning:

In [ ]:
from occlm.training.config import TrainingConfig, load_config
from occlm.training.sft_trainer import OCCTrainer

# Load training config
config = load_config("configs/training/sft_baseline.yaml")

# Initialize trainer
trainer = OCCTrainer(config)
trainer.load_model_and_tokenizer()

# Train
print("Starting training...")
metrics = trainer.train(
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

# Results
print(f"\nFinal metrics:")
for key, value in metrics.items():
    print(f"  {key}: {value:.4f}")

# Save
trainer.save_checkpoint("./checkpoints/sft-trained")
print("\nModel saved to ./checkpoints/sft-trained")


## OR Train with LoRA (Memory Efficient)

Parameter-efficient training:

In [ ]:
from occlm.training.lora_trainer import LoRATrainer

# Load LoRA config
config = load_config("configs/training/lora_baseline.yaml")

# Initialize trainer
trainer = LoRATrainer(config)
trainer.load_model_and_tokenizer()

# Train
print("Starting LoRA training...")
metrics = trainer.train(
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

print(f"Trainable parameters: {trainer.get_trainable_parameters_count():,}")

# Save adapters
trainer.save_lora_adapters("./checkpoints/lora-trained")
print("LoRA adapters saved to ./checkpoints/lora-trained")


## Hyperparameter Tuning

Experiment with different hyperparameters:

In [ ]:
from occlm.training.tracking import ExperimentTracker

# Track experiments
tracker = ExperimentTracker(project="taktkrone-i-tuning")

learning_rates = [1e-5, 2e-5, 5e-5]
batch_sizes = [16, 32]

results = {}

for lr in learning_rates:
    for bs in batch_sizes:
        run_name = f"lr-{lr:.0e}_bs-{bs}"
        
        config = TrainingConfig(
            learning_rate=lr,
            per_device_train_batch_size=bs,
        )
        
        trainer = OCCTrainer(config)
        trainer.load_model_and_tokenizer()
        
        with tracker:
            tracker.log_config(config.model_dump())
            metrics = trainer.train(train_dataset, val_dataset)
            tracker.log_metrics(metrics, step=0)
            results[run_name] = metrics

# Best config
best_run = max(results, key=lambda x: results[x].get('eval_accuracy', 0))
print(f"\nBest hyperparameters: {best_run}")


## Evaluate Trained Model

Test the trained model:

In [ ]:
# Load trained checkpoint
trainer = OCCTrainer.load_checkpoint("./checkpoints/sft-trained")

# Evaluate
eval_result = trainer.evaluate(val_dataset)
print(f"Evaluation results:")
print(f"  Accuracy: {eval_result['eval_accuracy']:.4f}")
print(f"  Loss: {eval_result['eval_loss']:.4f}")

# Create visualizations
trainer.create_loss_plot("loss_curve.png")
print("\nLoss curve saved to loss_curve.png")


## Push to Hugging Face Hub

Share your model:

In [ ]:
# Push to Hugging Face
trainer.push_to_hub(
    repo_id="your-username/taktkrone-i-fine-tuned",
    private=True  # or False for public
)

print("Model shared at https://huggingface.co/your-username/taktkrone-i-fine-tuned")
